In [17]:
import torch
from txtai import Embeddings
import pandas as pd
#from datasets import load_dataset
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import bm25s
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
import nltk
from FlagEmbedding import FlagReranker
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


2.14.1+cu126
True
NVIDIA GeForce RTX 4080 Laptop GPU


In [4]:
# Load ST model once for efficiency
_st_model = SentenceTransformer("all-MiniLM-L6-v2")


def evaluate(reference_answer: str, candidate_answer: str) -> dict:
    """
    Computes BLEU, ROUGE, METEOR, and Semantic Text Similarity.

    Args:
        reference_answer (str): Ground-truth answer.
        candidate_answer (str): Model-generated answer.

    Returns:
        dict: Evaluation metrics.
    """

    # --------------------
    # BLEU
    # --------------------
    reference_tokens = [word_tokenize(reference_answer.lower())]
    candidate_tokens = word_tokenize(candidate_answer.lower())

    bleu = sentence_bleu(
        reference_tokens,
        candidate_tokens,
        smoothing_function=SmoothingFunction().method1,
    )

    # --------------------
    # ROUGE
    # --------------------
    rouge = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True,
    )

    rouge_scores = rouge.score(reference_answer, candidate_answer)

    # --------------------
    # METEOR
    # --------------------
    meteor = meteor_score(
        [word_tokenize(reference_answer)],
        word_tokenize(candidate_answer),
    )

    # --------------------
    # Semantic Similarity (ST)
    # --------------------
    ref_emb = _st_model.encode([reference_answer])
    cand_emb = _st_model.encode([candidate_answer])

    semantic_similarity = cosine_similarity(ref_emb, cand_emb)[0][0]

    return {
        "BLEU": round(bleu, 4),
        "ROUGE-1": round(rouge_scores["rouge1"].fmeasure, 4),
        "ROUGE-2": round(rouge_scores["rouge2"].fmeasure, 4),
        "ROUGE-L": round(rouge_scores["rougeL"].fmeasure, 4),
        "METEOR": round(meteor, 4),
        "ST_SIMILARITY": round(float(semantic_similarity), 4),
    }

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1740.75it/s]


In [5]:
train = pd.read_json('pubmedqa/data/pqaa_train_set.json').T
train['id'] = train.index.astype(int)

test = train.iloc[700:1000]
train = train.iloc[0:700]
dev = pd.read_json('pubmedqa/data/pqaa_dev_set.json').T
contexts_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids_train = train.index.astype(str).tolist()
documents_train = list(zip(context_ids_train, contexts_train))

contexts_test = [" ".join(contexts) for contexts in test["CONTEXTS"]]
context_ids_test = test.index.astype(str).tolist()
documents_test = list(zip(context_ids_test, contexts_test))

print(train.head())
print(len(train))

                                                   QUESTION  \
22117569  Is an advance care planning model feasible in ...   
16097998  Is coeliac disease screening in risk groups ju...   
15223725  Does blood pressure change in treated hyperten...   
17274051  Metastatic carcinoma to the cervical nodes fro...   
27581329  Estimation of basal metabolic rate in Chinese:...   

                                                   CONTEXTS  \
22117569  [An effective advance care planning programme ...   
16097998  [The benefits of serologic screening for coeli...   
15223725  [To determine whether there are differences be...   
17274051  [The aim of the study was to evaluate the outc...   
27581329  [Measurement of basal metabolic rate (BMR) is ...   

                                                     LABELS  \
22117569  [BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...   
16097998                     [BACKGROUND, METHODS, RESULTS]   
15223725                      [OBJECTIVES, METHOD, RE

In [6]:
test

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER,id
27757987,Does the treatment of amblyopia normalise subf...,[Recent studies have found a choroidal thicken...,"[PURPOSE, METHODS, RESULTS]","[Adolescent, Amblyopia, Child, Child, Preschoo...",2017,no,no,no,Although a six-month treatment of amblyopia in...,27757987
24352924,Is portable ultrasonography accurate in the ev...,[The purpose of this study was to investigate ...,"[OBJECTIVE, DESIGN, PARTICIPANTS, INTERVENTION...","[Bone Nails, Cadaver, External Fixators, Femor...",2013,yes,yes,yes,Despite the statistical imparity of pin protru...,24352924
12790890,Is the cell death in mesial temporal sclerosis...,[Mesial temporal sclerosis (MTS) is characteri...,"[PURPOSE, METHODS, RESULTS]","[Actin Cytoskeleton, Adult, Age of Onset, Apop...",2003,maybe,no,maybe,These data suggest that either apoptosis is no...,12790890
22411435,Comparative safety of infliximab and etanercep...,"[Infliximab, a chimeric monoclonal anti-TNFα a...","[PURPOSE, METHODS, RESULTS]","[Adolescent, Adult, Age Factors, Aged, Antibod...",2012,yes,yes,yes,An increased risk of serious infections associ...,22411435
18670651,"Do the changes in the serum levels of IL-2, IL...",[Acute pancreatitis is the major complication ...,"[BACKGROUND, RESULTS]","[Cholangiopancreatography, Endoscopic Retrogra...",2008,yes,maybe,yes,The enhancement of serum TNFalpha and IL-6 lev...,18670651
...,...,...,...,...,...,...,...,...,...,...
17220021,Is there an increase in the incidence of gbs c...,[Group B Streptococci (GBS) asymptomatically c...,"[BACKGROUND, METHODS, RESULTS]","[Adolescent, Adult, Carrier State, Female, Hum...",2006,no,yes,yes,In the present study we found a significant in...,17220021
22075911,Is there a differential in the dental health o...,[Figures from the British Defence Dental Servi...,"[BACKGROUND AND AIM, METHOD, RESULTS]","[Adolescent, Cross-Sectional Studies, DMF Inde...",2011,yes,yes,yes,A significant difference in dental health betw...,22075911
10922093,Does open access endoscopy close the door to a...,[The use of open access endoscopy is increasin...,"[BACKGROUND, METHODS, RESULTS]","[Adult, Aged, Ambulatory Care, Endoscopy, Gast...",2000,yes,yes,yes,Patients undergoing open access procedures are...,10922093
25336163,Are interstitial fluid concentrations of merop...,[To describe the interstitial fluid (ISF) and ...,"[OBJECTIVES, PATIENTS AND METHODS, RESULTS]","[Aged, Anti-Bacterial Agents, Area Under Curve...",2015,no,no,no,This is the first known report of concurrent p...,25336163


In [7]:
# dense embeddings
dense_embeddings = Embeddings(path="neuml/pubmedbert-base-embeddings", content=True)
dense_embeddings.index(documents_train)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1740.76it/s]


In [8]:
# Test dense embeddings search
test_dense = dense_embeddings.search("Is an advance care planning model feasible in community palliative care?", limit = 50)

[d['id'] for d in test_dense]

['22117569',
 '18693227',
 '17329379',
 '8199520',
 '23719685',
 '22303473',
 '17355582',
 '11943048',
 '20629769',
 '21398266',
 '14551704',
 '12947068',
 '10158597',
 '9191526',
 '26578404',
 '20605051',
 '24014276',
 '18832500',
 '16827975',
 '25406780',
 '21658267',
 '21481154',
 '18800356',
 '18594195',
 '25571931',
 '12040336',
 '22188074',
 '8245806',
 '11334578',
 '12855939',
 '21252642',
 '16678696',
 '26864326',
 '23347337',
 '28143468',
 '15774570',
 '11247896',
 '19711462',
 '25103647',
 '22656647',
 '16968876',
 '22513023',
 '18619710',
 '16428354',
 '22491528',
 '15151701',
 '26561905',
 '23736032',
 '2224269',
 '17691856']

In [9]:
type(dense_embeddings)

txtai.embeddings.base.Embeddings

<h2>Sparse Embeddings</h2>

In [10]:


contexts_train = [" ".join(contexts) for contexts in train["CONTEXTS"]]
context_ids = train.index.to_numpy()
contexts_train_tokens = bm25s.tokenize(contexts_train, stopwords="en")

sparse_embeddings = bm25s.BM25(corpus=context_ids)
sparse_embeddings.index(contexts_train_tokens)

assert sparse_embeddings.scores["num_docs"] == len(train)
print(f"Indexed {len(contexts_train):,} contexts with BM25.")

Split strings:   0%|          | 0/700 [00:00<?, ?it/s]

Indexed 700 contexts with BM25.


<h2>RRF<h2>

In [ ]:
# collapsing the elemnts of the list in CONTEXTS into a singel string
corpus = {
    idx: " ".join(contexts)
    for idx, contexts in zip(
        train.index,
        train["CONTEXTS"]
    )
}

In [12]:
from typing import List, Dict

def reciprocal_rank_fusion(ranks: List[List[str]], k: int = 60) -> List[str]:
    """
    ranks: list of lists of document IDs from each retrieval method
    k: constant for rank discount
    """
    doc_scores = {doc: 0.0 for doc in set().union(*ranks)}
    for rank_list in ranks:
        for rank, doc in enumerate(rank_list, start=1):
            doc_scores[doc] += 1 / (rank + k)
    return sorted(doc_scores.keys(), key=lambda d: doc_scores[d], reverse=True)

# Example usage:
# list1 = ["docA", "docB", "docC"]
# list2 = ["docB", "docC", "docD"]
# final = reciprocal_rank_fusion([list1, list2], k=60)
# print(final)

def retrieve_and_fuse(query,dense_embeddings, sparse_embeddings, dense_limit=25, sparse_limit=25):
    dense_list= dense_embeddings.search(query, limit=dense_limit)
    dense_ids = [dict['id'] for dict in dense_list]
    query_tokens = bm25s.tokenize(query, stopwords="en")
    sparse_ids, _ = sparse_embeddings.retrieve(query_tokens, k=sparse_limit)
    sparse_ids = sparse_ids[0].tolist()
    fused_ids = reciprocal_rank_fusion([dense_ids, sparse_ids])
    fused_results = [
    {
        "doc_id": int(doc_id),
        "text": corpus[int(doc_id)]
    }
    for doc_id in fused_ids
]

    return fused_results
query = 'Is an advance care planning model feasible in community palliative care?'
fused_results = retrieve_and_fuse(query, dense_embeddings, sparse_embeddings)
#fused_list = list(set([int(doc["doc_id"]) for doc in fused_results]))
#train.loc[fused_list]
#fused_results = train.loc[fused_list]['CONTEXTS'].to_dict()# pass me to the next step for processing

In [13]:
fused_results

[{'doc_id': 22117569,
  'text': 'An effective advance care planning programme involves an organizational wide commitment and preparedness for health service reform to embed advance care planning into routine practice. Internationally, such programmes have been implemented predominantly in aged and acute care with more recent work in primary care. A multi-site action research was conducted over a 16-month period in 2007-2009 with three Victorian community palliative care services. Using mixed method data collection strategies to assess feasibility, we conducted a baseline audit of staff and clients; analysed relevant documents (client records, policies, procedures and quality improvement strategies) pre-implementation and post-implementation and conducted key informant interviews (n\u2003=\u20039). Three community palliative care services: one regional and two metropolitan services in Victoria, Australia. The services demonstrated that it was feasible to embed the Model into their organ

<h1>Ranking Layer</h1>

In [14]:
reranker = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=True
)

def rerank_documents(query, fused_results, reranker):
    """
    Rerank documents using BGE cross encoder.

    Parameters
    ----------
    query : str
    fused_results : list[dict]
        Must contain keys:
            - doc_id
            - text

    Returns
    -------
    list[dict]
        Sorted by reranker score
    """

    query_doc_pairs = [
        [query, doc["text"]]
        for doc in fused_results
    ]

    scores = reranker.compute_score(
        query_doc_pairs,
        normalize=True
    )

    for doc, score in zip(fused_results, scores):
        doc["rerank_score"] = float(score)

    reranked = sorted(
        fused_results,
        key=lambda x: x["rerank_score"],
        reverse=True,
    )

    return reranked


reranked_docs = rerank_documents(
    query=query,
    fused_results=fused_results,
    reranker=reranker
)

top_k = reranked_docs[:5]
print(top_k)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1658.91it/s]


[{'doc_id': 22117569, 'text': 'An effective advance care planning programme involves an organizational wide commitment and preparedness for health service reform to embed advance care planning into routine practice. Internationally, such programmes have been implemented predominantly in aged and acute care with more recent work in primary care. A multi-site action research was conducted over a 16-month period in 2007-2009 with three Victorian community palliative care services. Using mixed method data collection strategies to assess feasibility, we conducted a baseline audit of staff and clients; analysed relevant documents (client records, policies, procedures and quality improvement strategies) pre-implementation and post-implementation and conducted key informant interviews (n\u2003=\u20039). Three community palliative care services: one regional and two metropolitan services in Victoria, Australia. The services demonstrated that it was feasible to embed the Model into their organiz

In [16]:
top_k_ids = [candidate_ref['doc_id'] for candidate_ref in top_k]
top_k_text = [candidate_ref['text'] for candidate_ref in top_k]
top_k_rerank_scores = [candidate_ref['rerank_score'] for candidate_ref in top_k]
train.loc[top_k_ids]

,QUESTION,CONTEXTS,LABELS,MESHES,YEAR,reasoning_required_pred,reasoning_free_pred,final_decision,LONG_ANSWER,id
22117569,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...,22117569
22117569,Is an advance care planning model feasible in ...,[An effective advance care planning programme ...,"[BACKGROUND, METHODS, SETTINGS AND PARTICIPANT...","[Adult, Advance Care Planning, Advance Directi...",2012,yes,yes,yes,An advance care planning model is feasible for...,22117569
17329379,Are the GFRUP's recommendations for withholdin...,[To evaluate feasibility of the guidelines of ...,"[OBJECTIVE, DESIGN, SETTING, PATIENTS, RESULTS]","[Child, Preschool, Critical Illness, Decision ...",2007,yes,yes,yes,GFRUPs procedure was applicable in most cases....,17329379
21658267,"Do improvements in outreach, clinical, and fam...",[There are three main service delivery channel...,"[BACKGROUND, METHODS, RESULTS]","[Adolescent, Adult, Africa, Asia, Caribbean Re...",2011,yes,yes,yes,Our findings support the importance of increas...,21658267
21658267,"Do improvements in outreach, clinical, and fam...",[There are three main service delivery channel...,"[BACKGROUND, METHODS, RESULTS]","[Adolescent, Adult, Africa, Asia, Caribbean Re...",2011,yes,yes,yes,Our findings support the importance of increas...,21658267


<h1>Augment generator with top _k</h1>

- input structure: System prompt, User Query, Documents <br>
1) Set up a system prompt, 
2) Tokenize top_k documents and query

To-do- filter out low scoring top scores

In [18]:
# Get model and tokenizer 
model_name = "Qwen/Qwen2.5-7B-Instruct"

def initialize_model_and_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Configure BitsAndBytes for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return model, tokenizer

model, tokenizer = initialize_model_and_tokenizer(model_name)

Loading weights: 100%|██████████| 339/339 [00:22<00:00, 15.07it/s]


In [27]:
def structure_message(query, top_docs):
    """
    Structures a message for the model to answer a biomedical question using the provided context.

    Args:
        query (str): The biomedical question.
        top_docs (list[str]): The top documents providing context. The list is expected to be ordered by relevance and 

    Returns:
        list[dict]: A list of messages structured for the model.
    """
    context = "\n\n".join(top_docs)
    messages = [
    {
        "role": "system",
        "content": (
            "You are a biomedical question answering assistant. "
            "Answer using only the provided context."
        )
    },
    {
    "role": "user",
    "content": f"""
        Question:
        {query}

        Context:
        {context}
        """
    }
    ]
    return messages
my_message = structure_message(query, top_k_text)



In [37]:
def serve_model(model, tokenizer, query, top_docs):
    message = structure_message(query, top_docs)
    text = tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1,)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [46]:
query_id = train[train['QUESTION'] == query].index[0]
output = serve_model(model, tokenizer, query, top_k_text)
true_response = train.loc[query_id, 'LONG_ANSWER']
candidate_response = output.split('\nassistant\n')[-1]
print("Candidate Response:", candidate_response)
print("True Response:", true_response)


Candidate Response: Yes, an advance care planning model is feasible in community palliative care. The study involving three Victorian community palliative care services demonstrated that embedding advance care planning into their organizational structures was feasible over a 16-month period. Key findings include the importance of advance care planning conversations and family involvement, adaptation of the concept of community to widen the model's impact, and changes to quality audit processes to consolidate the model into routine practice.
True Response: An advance care planning model is feasible for community palliative care services. Quality audit processes are an essential component of the Model with documentation of advance care planning discussion established as an important outcome measure.


In [47]:
score_dict = evaluate(candidate_response, true_response)

In [48]:

score_dict

{'BLEU': 0.1046,
 'ROUGE-1': 0.3964,
 'ROUGE-2': 0.2936,
 'ROUGE-L': 0.2883,
 'METEOR': 0.2428,
 'ST_SIMILARITY': 0.8648}